In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:  # aqui mantemos menor que o limite
            return True
        return False
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [4]:
analize = Analizer(0.8)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
5,model_5_1_2,0.796336,-0.229313,0.598710,0.778077,0.811348,0.298228,1.800102,0.347618,0.251744,0.299681,1.065959,0.546103,0.301724,0.569351,36.419794,57.140683,"Hidden Size=[4], regularizer=0.2, learning_rat..."
7,model_1_4_24,0.793887,-0.068718,0.904188,0.548125,0.782254,0.301814,1.564939,0.197660,0.486534,0.342097,1.467790,0.549376,1.824451,0.572764,62.395890,98.962164,"Hidden Size=[3, 4], regularizer=0.2, learning_..."
8,model_5_1_3,0.793457,-0.295319,0.219871,0.806442,0.718182,0.302444,1.896754,0.675789,0.219568,0.447678,2.830065,0.549949,0.291852,0.573362,36.391718,57.112607,"Hidden Size=[4], regularizer=0.2, learning_rat..."
9,model_33_7_24,0.793357,0.001276,0.552138,0.695779,0.592328,0.302590,1.462446,1.085857,0.252597,0.669227,4.372165,0.550082,1.291731,0.573500,84.390753,134.364662,"Hidden Size=[10], regularizer=0.2, learning_ra..."
10,model_1_4_23,0.793190,-0.071308,0.905317,0.562571,0.787945,0.302835,1.568732,0.195330,0.470980,0.333155,1.477267,0.550305,1.827241,0.573732,62.389133,98.955408,"Hidden Size=[3, 4], regularizer=0.2, learning_..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
580,model_27_4_16,0.746075,-0.171381,0.488399,0.906301,0.719338,0.371826,1.715271,0.920777,0.110632,0.515705,1.176762,0.609775,1.053458,0.635735,277.978660,446.183524,"Hidden Size=[9, 10], regularizer=0.05, learnin..."
581,model_23_1_11,0.746038,-0.010759,0.682948,0.949571,0.856306,0.371880,1.480069,0.294499,0.069524,0.182011,2.139372,0.609820,1.015124,0.635781,855.978367,1376.438344,"Hidden Size=[14, 24], regularizer=0.05, learni..."
583,model_7_9_10,0.745857,-0.575716,0.838802,0.962709,0.898715,0.372146,2.307344,0.325926,0.070546,0.198236,2.641403,0.610038,1.321023,0.636008,87.976939,140.388600,"Hidden Size=[4, 5], regularizer=0.05, learning..."
585,model_23_2_14,0.745803,-0.046606,0.648193,0.739227,0.669329,0.372224,1.532560,0.858275,0.157502,0.507889,0.885405,0.610101,1.067041,0.636075,231.976521,372.147241,"Hidden Size=[8, 9], regularizer=0.05, learning..."
